# 1. Initialization & Environment Setup
This section imports necessary libraries, establishes strict determinism flags for reproducible GPU training, and defines global hyperparameters including trading costs (slippage, spread, fees) and starting capital.

In [ ]:
import torch as th
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import datetime as dt
from torchinfo import summary
from pprint import pprint
import os
from typing import Union, Callable, Optional
from scipy.stats import spearmanr, pearsonr
import seaborn as sns
from copy import deepcopy
import cvxpy as cp
import pickle
import glob
from collections import defaultdict
import json
import itertools
from sklearn.isotonic import IsotonicRegression


from utility import (
    build_stock_dataset, compute_intraday_features, Market,
    ALL_TICKERS, DJ30_TICKERS, NUM_STOCKS, 
    Horizon, Offset, WindowSize, PredStep, 
    train_days, step_days, train_start_date, 
    IDX_DAY_COUNTER, IDX_LOGRET_CLOSE, 
    batchsize, eval_batchsize, epochs, iters, patience,
    dropout, num_layers, layer_dim, feature_indicies,
    BackboneInputType, ModelOutputType, backbone_input_type, model_output_type,
    lr_start, lr_end, return_loss_coeff, drift_loss_coeff, 
    device, l2_norm, burn_in_epochs, markowitz_reg
)

In [ ]:
# ============================
# Hyperparameters (in utility.py)
# ============================

# --- Determinism flags ---
th.backends.cudnn.deterministic = True
th.backends.cudnn.benchmark = False
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # CUDA matmul determinism
th.use_deterministic_algorithms(True)


# Costs Constant and starting amount.
start_capital= 1000_000
slippage_bps=0.000
spread_bps=0.000
fee_bps=0.0000

# 2. Data Ingestion & Scale-Invariant Preprocessing
Loads the raw DJ30 and ETF price data, computes scale-invariant intraday features (e.g., log returns, normalized oscillators, localized volatility), and strictly aligns the temporal indices to handle missing API data without lookahead bias.

In [ ]:
# DJ30 tickers
# raw_dicts = {}
# for tic in ALL_TICKERS:
#     print(f'===== Starting {tic} =====')
#     raw_dicts[tic] = build_stock_dataset(tic, Market.US, Horizon, Offset)

In [ ]:
# Read Stocks Info
with open(Path.cwd().parent / 'Temp' / 'us_stocks.pkl', "rb") as f:
    raw_dicts = pickle.load(f)

# Save Stocks Info
with open(Path.cwd().parent / 'Temp' / 'us_stocks.pkl', "wb") as f:
    pickle.dump(raw_dicts, f)


In [ ]:
""" Create features and drop NaN rows"""

feature_dicts, close_dicts = compute_intraday_features(raw_dicts)

ref_ticker = ALL_TICKERS[0]
raw_df = raw_dicts[ref_ticker]
feat_df = feature_dicts[ref_ticker] # Pre-cleaning feature DataFrame

# Identify and drop initial NaN rows (typically caused by rolling windows/lags)
first_valid_idx = feat_df.notna().all(axis=1).idxmax()

# Build a boolean mask: True for rows occurring on or after the first valid index
valid_mask = feat_df.index >= first_valid_idx

# Apply this temporal mask across ALL tickers to ensure strict index alignment
for sym in ALL_TICKERS:
    feature_dicts[sym] = feature_dicts[sym].loc[valid_mask]
    close_dicts[sym]   = close_dicts[sym].loc[valid_mask]

# Generate a Unified Data Summary
clean_features = [col.split('_', 1)[-1] for col in feature_dicts[ref_ticker].columns]
rows_dropped = len(feat_df) - len(feature_dicts[ref_ticker])

print(f"--- Unified Data Summary (Ref: {ref_ticker}) ---")
print(f"Raw Shape:           {raw_df.shape[0]} rows x {raw_df.shape[1]} columns")
print(f"Final Cleaned Shape: {feature_dicts[ref_ticker].shape[0]} rows x {feature_dicts[ref_ticker].shape[1]} columns")
print(f"-> Dropped {rows_dropped} invalid rows due to feature lookback windows.\n")

print(f"Extracted Features ({len(clean_features)} total):")
pprint(clean_features, compact=True)

print(f"\nFinal Valid Index Preview:")
print(feature_dicts[ref_ticker].index[:5])

In [ ]:
def shuffle_features(feature_dicts, feature_to_shuffle):
    """
    Shuffles a specific feature (e.g., 'raw_effr') across the time axis 
    for all tickers while keeping other features intact.
    """
    shuffled_dicts = {}
    
    for tic, df in feature_dicts.items():
        shuffled_df = df.copy()
        col_name = f"{tic}_{feature_to_shuffle}"
        
        shuffled_df[col_name] = np.random.permutation(shuffled_df[col_name].values)
        # shuffled_df[col_name] = np.zeros_like(shuffled_df[col_name].to_numpy())
        shuffled_dicts[tic] = shuffled_df
        
    return shuffled_dicts

def swap_features(feature_dicts, feat_a="raw_snr", feat_b="raw_effr"):
    """
    Swaps the contents of two feature columns for all tickers.
    Tests if the model is sensitive to the specific 'meaning' of the ratio.
    """
    swapped_dicts = {}
    
    for tic, df in feature_dicts.items():
        swapped_df = df.copy()
        col_a = f"{tic}_{feat_a}"
        col_b = f"{tic}_{feat_b}"
        
        # Perform the swap using a temporary hold
        temp_values = swapped_df[col_a].copy()
        swapped_df[col_a] = swapped_df[col_b]
        swapped_df[col_b] = temp_values
        
        swapped_dicts[tic] = swapped_df
        
    return swapped_dicts

def ablate_features(feature_dicts, features_to_kill, mode='zero'):
    """
    Forces the model to ignore specific information.
    
    Args:
        feature_dicts: Dict of DataFrames {ticker: df}
        features_to_kill: List of base feature names (e.g. ['rsi_14', 'vwap_dist'])
        mode: 'zero' to fill with 0s, 'mean' to fill with the column mean.
    """
    ablated_dicts = {}
    
    for tic, df in feature_dicts.items():
        temp_df = df.copy()
        
        for feat in features_to_kill:
            # Construct the full column name based on your prefix logic
            col_name = f"{tic}_{feat}"
            
            if col_name in temp_df.columns:
                if mode == 'zero':
                    temp_df[col_name] = 0.0
                elif mode == 'mean':
                    temp_df[col_name] = temp_df[col_name].mean()
            else:
                print(f"Warning: {col_name} not found in dataframe.")
                
        ablated_dicts[tic] = temp_df
        
    return ablated_dicts


<!-- ## Train Test Sets Creation -->

# 3. Walk-Forward Dataset Generation
Transforms the scale-invariant features into a structured, sequential tensor dataset (`B, K, T, D`). It establishes the chronological Walk-Forward Optimization (WFO) folds, ensuring strict temporal separation between Train, Validation, and Test periods to simulate real-world trading conditions.

In [ ]:
"""
Convert individual tic's dataframes into a combined dataset for training. Window Size is T, the number of 
timesteps info of the past to include,  and predstep is the prediction step in the future. 
"""

def make_multisymbol_window_dataset(feature_dicts, close_dicts, symbols, window, pred_step):
    # --- 1. PRE-STACK GLOBAL DATA (FLOAT32) ---
    ref_df = feature_dicts[symbols[0]]
    # Force float32 during stacking to prevent 'object' dtypes
    all_features = np.stack([feature_dicts[sym].to_numpy(dtype=np.float32) for sym in symbols], axis=1)
    all_prices   = np.stack([close_dicts[sym].to_numpy().flatten().astype(np.float32) for sym in symbols], axis=1)

    # --- 2. IDENTIFY TARGET INDICES ---
    full_indices = np.arange(len(ref_df))
    # We only want targets that have enough history to form a complete window + prediction step
    eligible_indices = full_indices[full_indices >= (window + pred_step - 1)]
    
    B = len(eligible_indices)
    K, D = len(symbols), all_features.shape[2]

    # --- 3. PRE-ALLOCATE OUTPUTS ---
    X_arr          = np.zeros((B, K, window, D), dtype=np.float32)
    logret_arr     = np.zeros((B, K), dtype=np.float32)
    close_curr_arr = np.zeros((B, K), dtype=np.float32)
    close_next_arr = np.zeros((B, K), dtype=np.float32)

    # --- 4. EXPLICIT WINDOWING LOOP ---
    for i, target_idx in enumerate(eligible_indices):
        # target_idx: The moment the "next" price/return happens
        # curr_idx: The moment we are standing in to make the prediction
        curr_idx = target_idx - pred_step
        
        # Window: History leading up to and including 'curr_idx'
        # e.g., if window=5, we take [curr-4, curr-3, curr-2, curr-1, curr]
        window_start = curr_idx - window + 1
        window_end   = curr_idx + 1 # +1 for inclusive numpy slicing
        
        # Features (K, window, D)
        X_arr[i] = np.swapaxes(all_features[window_start : window_end], 0, 1)
        
        # Targets (All K symbols)
        logret_arr[i]      = all_features[target_idx, :, IDX_LOGRET_CLOSE]
        
        # Prices
        close_curr_arr[i]  = all_prices[curr_idx]
        close_next_arr[i]  = all_prices[target_idx]

    # Make the day counter relative
    X_arr[:, :, :, IDX_DAY_COUNTER] -= X_arr[:, :, -1:, IDX_DAY_COUNTER]

    return {
        "index": ref_df.index[eligible_indices],
        "X": X_arr,
        "logret_close": logret_arr,
        "close_curr": close_curr_arr,
        "close_next": close_next_arr
    }

def check_dataset_for_nans(name, dataset, symbols=None, feature_names=None):
    print(f"=== Checking {name} dataset ===")
    for key, arr in dataset.items():
        # Ensure we are working with torch for consistency, or keep as numpy
        arr_th = th.from_numpy(arr) if isinstance(arr, np.ndarray) else arr
        
        has_nan = th.isnan(arr_th).any().item()
        has_inf = th.isinf(arr_th).any().item()
        
        if has_nan or has_inf:
            print(f"{key:12s} | shape {arr_th.shape} | NaNs: {has_nan} | Infs: {has_inf}")
            
            # Find the indices of problematic values
            # (mask returns a tuple of coordinate arrays)
            bad_mask = th.isnan(arr_th) | th.isinf(arr_th)
            indices = th.nonzero(bad_mask)
            
            # X_arr usually has shape (B, K, window, D)
            if key == "X" and indices.shape[0] > 0:
                # We'll just show the first few to avoid flooding the console
                num_to_show = min(5, indices.shape[0])
                print(f"  First {num_to_show} problematic locations [Batch, Symbol, Window, Feature]:")
                for i in range(num_to_show):
                    idx = indices[i].tolist()
                    sym_name = symbols[idx[1]] if symbols else idx[1]
                    feat_name = feature_names[idx[3]] if feature_names else idx[3]
                    print(f"    - Batch {idx[0]}, Symbol: {sym_name}, Feature: {feat_name}")
            
            # Other arrays (logret, etc.) usually (B, K)
            elif indices.shape[0] > 0:
                num_to_show = min(5, indices.shape[0])
                print(f"  First {num_to_show} problematic locations [Batch, Symbol]:")
                for i in range(num_to_show):
                    idx = indices[i].tolist()
                    sym_name = symbols[idx[1]] if symbols else idx[1]
                    print(f"    - Batch {idx[0]}, Symbol: {sym_name}")
    print()

def compute_r_values(start: dt.date, end:dt.date):
    global feature_dicts, ALL_TICKERS
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    r_values = []
    for sym in ALL_TICKERS:
        df = feature_dicts[sym]
        mask = (df.index >= start) & (df.index < end)
        df_slice = df.loc[mask]

        logret_close = df_slice[f"{sym}_logret_close"].to_numpy(dtype=np.float64)[:-1]
        logret_open  = df_slice[f"{sym}_logret_open"].to_numpy(dtype=np.float64)[:-1]

        r_val = float(np.mean(logret_close * logret_open) /
                      np.mean(logret_close * logret_close))
        r_values.append(r_val)

    return r_values

# Generate the Master Dataset
master_dataset = make_multisymbol_window_dataset(
    feature_dicts, 
    close_dicts, 
    ALL_TICKERS, 
    WindowSize, 
    PredStep, 
)
master_idx = master_dataset['index']
# 2. Global Integrity Check
print("--- Initializing Global Data Integrity Check ---")
check_dataset_for_nans("Master_Dataset", { k:master_dataset[k] for k in  master_dataset.keys() if k !='index'}, symbols=ALL_TICKERS)

# --- 1. Extract the Calendar of Trading Days ---
# Get unique dates from the DatetimeIndex and sort them
trading_calendar = pd.Series(master_dataset['index'].normalize().unique()).sort_values()
mask = trading_calendar >= train_start_date
trading_calendar = trading_calendar[mask].values

num_trading_days = len(trading_calendar)

all_folds_data = []
current_start_idx = 0

# --- 2. Slicing Loop based on Trading Day Index ---
while True:
    # Define integer positions in the trading_calendar
    train_end_idx = current_start_idx + train_days
    val_end_idx   = train_end_idx + step_days
    test_end_idx  = val_end_idx + step_days
    
    # Exit Condition: If we can't complete a full Test window, we stop
    if test_end_idx > num_trading_days:
        print(f"\n[Finished] Stopping: Not enough trading days left for a full test slice.")
        break

    # Translate integer positions to actual Dates
    d_train_start = trading_calendar[current_start_idx]
    d_train_end   = trading_calendar[train_end_idx] # This date belongs to VAL (exclusive for TRAIN)
    d_val_end     = trading_calendar[val_end_idx]   # This date belongs to TEST
    d_test_end    = trading_calendar[test_end_idx-1] # The last inclusive date of TEST

    # --- 3. Create Masks against the Master Index ---
    # Since d_train_end is the start of the next period, we use < 
    # to keep days mutually exclusive.
    m_train = (master_idx >= d_train_start) & (master_idx < d_train_end)
    m_val   = (master_idx >= d_train_end)   & (master_idx < d_val_end)
    m_test  = (master_idx >= d_val_end)     & (master_idx <= np.datetime64(d_test_end) + np.timedelta64(1, 'D'))

    # --- 4. Slice the Master Dataset ---
    f_train = {k: (v[m_train] if isinstance(v, (np.ndarray, pd.Index)) else v) for k, v in master_dataset.items()}
    f_val   = {k: (v[m_val]   if isinstance(v, (np.ndarray, pd.Index)) else v) for k, v in master_dataset.items()}
    f_test  = {k: (v[m_test]  if isinstance(v, (np.ndarray, pd.Index)) else v) for k, v in master_dataset.items()}

    # 5. Compute R-values (on training date range)
    r = compute_r_values(d_train_start, d_train_end)

    all_folds_data.append({
        "meta": {
            "fold_no": len(all_folds_data) + 1,
            "train_range": (d_train_start, d_train_end),
            "val_range": (d_train_end, d_val_end),
            "test_range": (d_val_end, d_test_end)
        },
        "train": f_train, "val": f_val, "test": f_test, "r": r
    })

    # --- 6. Detailed Tabular Output ---
    print(f"FOLD {len(all_folds_data)} SUMMARY (Trading Day Based)")
    print(f"{'Phase':<10} | {'Start Date':<12} | {'End (Excl)':<12} | {'Samples':<8}")
    print(f"{'-'*55}")
    print(f"{'TRAIN':<10} | {pd.to_datetime(d_train_start).date()} | {pd.to_datetime(d_train_end).date()} | {m_train.sum()}")
    print(f"{'VAL':<10} | {pd.to_datetime(d_train_end).date()} | {pd.to_datetime(d_val_end).date()} | {m_val.sum()}")
    print(f"{'TEST':<10} | {pd.to_datetime(d_val_end).date()} | {pd.to_datetime(d_test_end).date()} | {m_test.sum()}")
    print("\n" + "="*60 + "\n")

    # 7. Roll forward by step size (in trading days)
    current_start_idx += step_days


# 4. Core Neural Architecture: Transformers & RVE
Defines the custom spatial-temporal modules. Includes Rotary Positional Embeddings (RoPE) for time-series, a Cross-Attention block for macroeconomic conditioning, and the dynamic Return Volatility Estimator (RVE) / SuperNetwork to facilitate the information bottleneck ablation study.

In [ ]:
class MultiheadAttentionRoPE(nn.Module):
    """
    DeBERTa-inspired Disentangled Attention for financial time-series.
    """
    def __init__(self, embed_dim, num_heads, max_seq_len=2048, dropout=0.0, disentangled=False):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        assert num_heads % 2 == 0, "num_heads must be even to split between P and T networks"
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.dim_head = embed_dim // num_heads
        self.max_seq_len = max_seq_len
        self.disentangled = disentangled

        # Output of each projection is embed_dim // 2. 
        # Total parameters exactly equal standard Q, K, V projections.
        half_embed = embed_dim // 2
        
        # Network P (Price Projections)
        self.W_q_p = nn.Linear(embed_dim, half_embed)
        self.W_k_p = nn.Linear(embed_dim, half_embed)
        self.W_v_p = nn.Linear(embed_dim, half_embed)
        
        # Network T (Time Projections)
        self.W_q_t = nn.Linear(embed_dim, half_embed)
        self.W_k_t = nn.Linear(embed_dim, half_embed)
        self.W_v_t = nn.Linear(embed_dim, half_embed)

        self.W_o = nn.Linear(embed_dim, embed_dim)
        self.softmax_dropout = nn.Dropout(dropout)

        # Precompute RoPE constants
        half_dim = self.dim_head // 2
        freq_seq = th.arange(half_dim, dtype=th.float32)
        inv_freq = (10000 ** -(freq_seq / half_dim))
        pos = th.arange(max_seq_len, dtype=th.float32)
        freqs = th.outer(pos, inv_freq)
        
        self.register_buffer("cos_cached", freqs.cos()[None, :, None, :], persistent=False)
        self.register_buffer("sin_cached", freqs.sin()[None, :, None, :], persistent=False)

    def _rope(self, x, seq_len):
        """ Standard RoPE rotation - provided by user, do not change """
        B, T, H, Dh = x.shape
        cos = self.cos_cached[:, :seq_len]
        sin = self.sin_cached[:, :seq_len]
        Dh_even = Dh - (Dh % 2)
        x_even = x[..., :Dh_even:2]
        x_odd  = x[..., 1:Dh_even:2]
        x_rot_even = x_even * cos - x_odd * sin
        x_rot_odd  = x_even * sin + x_odd * cos
        x_rot = th.empty_like(x)
        x_rot[..., :Dh_even:2] = x_rot_even
        x_rot[..., 1:Dh_even:2] = x_rot_odd
        if Dh % 2 == 1:
            x_rot[..., -1] = x[..., -1]
        return x_rot

    def forward(self, x, xt=None):
        """
        x: Price features [B, T, D]
        xt: Absolute time features [B, T, D] (Optional)
        """
        B, T, D = x.shape
        H, Dh = self.num_heads, self.dim_head
        
        if self.disentangled:
            assert xt is not None
            # Reduce number of heads by 2 for the split networks
            H_half = H // 2
            
            # 1. Price Network (Outputs embed_dim // 2 -> H_half heads)
            q_p = self.W_q_p(x).view(B, T, H_half, Dh)
            k_p = self.W_k_p(x).view(B, T, H_half, Dh)
            v_p = self.W_v_p(x).view(B, T, H_half, Dh)
            
            # 2. Time Network (Outputs embed_dim // 2 -> H_half heads)
            q_t = self.W_q_t(xt).view(B, T, H_half, Dh)
            k_t = self.W_k_t(xt).view(B, T, H_half, Dh)
            v_t = self.W_v_t(xt).view(B, T, H_half, Dh)

            # 3. Apply RoPE only to Price
            q_p = self._rope(q_p, T)
            k_p = self._rope(k_p, T)

            # 4. Disentangled Interaction (H_half heads)
            q_inter = q_p + q_t
            k_inter = k_p + k_t
            
            # To get output of kv as full embed_dim, V concatenates v_p and v_t (H heads total).
            # We duplicate the H_half queries and keys to match the H heads of V.
            q = th.cat([q_inter, q_inter], dim=2)
            k = th.cat([k_inter, k_inter], dim=2)
            v = th.cat([v_p, v_t], dim=2)
            
            scale = (4 * Dh) ** 0.5
        else:
            # Mode: Standard Attention. 
            # Concatenate P and T layers to reconstruct full D -> D projections
            q = th.cat([self.W_q_p(x), self.W_q_t(x)], dim=-1).view(B, T, H, Dh)
            k = th.cat([self.W_k_p(x), self.W_k_t(x)], dim=-1).view(B, T, H, Dh)
            v = th.cat([self.W_v_p(x), self.W_v_t(x)], dim=-1).view(B, T, H, Dh)
            
            q = self._rope(q, T)
            k = self._rope(k, T)
            scale = Dh ** 0.5

        # Standard Attention Mechanism
        q = q.transpose(1, 2) # [B, H, T, Dh]
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        attn_scores = th.matmul(q, k.transpose(-2, -1)) / scale
        attn_weights = th.nn.functional.softmax(attn_scores, dim=-1)
        attn_weights = self.softmax_dropout(attn_weights)

        out = th.matmul(attn_weights, v)
        out = out.transpose(1, 2).reshape(B, T, D)
        
        return self.W_o(out)
    
class TransformerEncoderLayerRoPE(nn.Module):
    def __init__(
        self,
        d_model: int,
        nhead: int,
        dim_feedforward: int = 2048,
        dropout: float = 0.1,
        activation = F.relu,
        layer_norm_eps: float = 1e-5,
        norm_first: bool = False,
        bias: bool = True,
        device=None,
        dtype=None,
        disentangled = False,
    ) -> None:
        factory_kwargs = {"device": device, "dtype": dtype}
        super().__init__()
        self.disentangled=disentangled
        self.self_attn = MultiheadAttentionRoPE(
            d_model,
            nhead,
            dropout=dropout,
            disentangled=self.disentangled
        )

        # Feedforward network
        self.linear1 = nn.Linear(d_model, dim_feedforward, bias=bias, **factory_kwargs)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model, bias=bias, **factory_kwargs)

        # Normalization and dropout
        self.norm_first = norm_first
        self.norm1 = nn.LayerNorm(d_model, eps=layer_norm_eps, bias=bias, **factory_kwargs)
        self.norm2 = nn.LayerNorm(d_model, eps=layer_norm_eps, bias=bias, **factory_kwargs)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.activation = activation

    def forward(
        self,
        x: th.Tensor,
        xt:th.Tensor = None,
    ) -> th.Tensor:
        # Pre-norm or post-norm depending on flag
        if self.disentangled:
            if self.norm_first:
                x = x + self.dropout1(self.self_attn(x,xt))
                x = x + self._ff_block(self.norm2(x))
            else:
                x = self.norm1(x + self.dropout1(self.self_attn(x,xt)))
                x = self.norm2(x + self._ff_block(x))

        else:
            if self.norm_first:
                x = x + self.dropout1(self.self_attn(x))
                x = x + self._ff_block(self.norm2(x))
            else:
                x = self.norm1(x + self.dropout1(self.self_attn(x)))
                x = self.norm2(x + self._ff_block(x))
        return x

    def _sa_block(
        self,
        x: th.Tensor,
    ) -> th.Tensor:
        out = self.self_attn(x)
        return self.dropout1(out)

    def _ff_block(self, x: th.Tensor) -> th.Tensor:
        return self.dropout2(self.linear2(self.dropout(self.activation(self.linear1(x)))))
    
class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead, dropout, dim_feedforward=2048):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        
        # The "Storage/Processing" MLP
        self.mlp = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(), # Smooth non-linearity for financial data
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout)
        )
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, context):
        """
        x: (B, N, E) - Stock tokens (Query)
        context: (B, M, E) - Global tokens (Key/Value)
        """
        # 1. Multi-Head Attention + Residual + Norm (Post-Norm Style)
        attn_out, _ = self.attn(query=x, key=context, value=context)
        x = self.norm1(x + attn_out) 
        
        # 2. MLP + Residual + Norm (The "Digestion" phase)
        # This is where the interaction between stock and global context is processed
        x = self.norm2(x + self.mlp(x)) 
        
        return x
 



In [ ]:
# Designing Networks

class ReturnVolatilityEstimateNetwork(nn.Module):
    def __init__(self, D, dropout, num_layers, layer_dim):
        super().__init__()
        self.D = D
        self.E = layer_dim
        # expansion RVE
        self.W = nn.Parameter(th.empty(self.E, D))
        nn.init.xavier_uniform_(self.W)
        self.b = nn.Parameter(th.zeros(self.E))
        
        self.lstm = nn.LSTM(input_size=self.E, hidden_size=self.E,
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0,
                            bidirectional=False)

    def forward(self, x, logret_close):
        B, K, T, D = x.shape

        x_flat = x.view(B * K, T, D)
        lr_flat = logret_close.view(B * K, T)

        # projection
        x_tilde = th.tanh(th.matmul(x_flat, self.W.t()) + self.b) 

        # LSTM + Attention logic remains same, but operates on enriched features
        h_seq, _ = self.lstm(x_tilde)
        h_T = h_seq[:, -1, :].unsqueeze(1)
        logits = (h_seq * h_T).sum(dim=2)
        alphas = th.nn.functional.softmax(logits.clamp(min=-50, max=50), dim=1)

        musigma = (alphas * lr_flat).sum(dim=1)
        diff = lr_flat - musigma.unsqueeze(1)
        var = (alphas * (diff * diff)).sum(dim=1)
        sigma = th.sqrt(var.clamp(min=1e-8))

        return musigma.view(B, K), sigma.view(B, K)
    
class MuBackBone(nn.Module):
    def __init__(self, indim, output_type, num_stocks, dropout, num_layers, layer_dim):
        super().__init__()
        self.num_stocks = num_stocks
        self.indim = indim
        self.output_type = output_type
        d_model = layer_dim
        nhead = d_model // 32

        # 1. Temporal Encoder (Sequential per-stock history)
        self.input_proj = nn.Linear(self.indim, d_model)
        
        self.time_layers = nn.ModuleList([
            TransformerEncoderLayerRoPE(
                d_model=d_model, 
                nhead=nhead, 
                dropout=dropout, 
                disentangled=False
            ) for _ in range(num_layers)
        ])

        # 2. Global Context Processor (Stocks attend to market globals)
        self.global_context_processor = CrossAttentionBlock(
            d_model=d_model, nhead=nhead, dropout=dropout
        )

        # 3. Cross-Token Encoder (Stocks attend to each other)
        self.cross_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model, 
                nhead=nhead, 
                dropout=dropout, 
                batch_first=True
            ) for _ in range(num_layers)
        ])

        # 4. Dynamic Output Heads
        if self.output_type == ModelOutputType.logreturn:
            self.error_head = nn.Linear(d_model, 1)
        elif self.output_type == ModelOutputType.rank:
            self.rank_head = nn.Linear(d_model, 1)
        elif self.output_type == ModelOutputType.effr_vol:
            self.effr_head = nn.Linear(d_model, 1)
            self.vol_head = nn.Linear(d_model, 1)

    def forward(self, x:th.Tensor):
        B, K, T, _ = x.shape

        # --- 1. Reshaping ---
        x_flat = x.reshape(B * K, T, self.indim)    

        # --- 2. Temporal Encoding (Price & Time Entangled via indim) ---
        h = self.input_proj(x_flat) 
        for layer in self.time_layers:
            h = layer(h) 

        # Pooling: Pull the representation of the most recent tick
        tokens = h[:, -1, :].view(B, K, -1) # (B, K, d_model)

        # --- 4. Market Conditioning (Cross-Attention) ---
        # Stocks look at market-wide global features
        stock_tokens = self.global_context_processor(
            tokens[:, :self.num_stocks, :],     # stocks
            tokens[:, self.num_stocks:, :] )    # global tokens

        # --- 5. Spatial Encoding (Stock-to-Stock) ---
        # Using the manual loop for the cross-encoder
        cross_h = stock_tokens
        for layer in self.cross_layers:
            cross_h = layer(cross_h)

        # --- 6. Output (ALWAYS returns a tuple) ---
        if self.output_type == ModelOutputType.logreturn:
            return (self.error_head(cross_h).squeeze(-1),) 
            
        elif self.output_type == ModelOutputType.rank:
            return (self.rank_head(cross_h).squeeze(-1),)
            
        elif self.output_type == ModelOutputType.effr_vol:
            effr_out = self.effr_head(cross_h).squeeze(-1)
            vol_out = self.vol_head(cross_h).squeeze(-1)
            return (effr_out, vol_out)

class ReVolSuperNetwork(nn.Module):
    def __init__(self, input_type: BackboneInputType, output_type: ModelOutputType, 
                 r_est: np.ndarray, feature_indices: dict, 
                 num_stocks: int, dropout: float, num_layers: int, layer_dim: int, D: int):
        super().__init__()
        self.idx = feature_indices
        self.input_type = input_type
        self.output_type = output_type
        
        # Determine Backbone Input Dimension based on Enum
        backbone_dim = 4 if self.input_type == BackboneInputType.revol_only else D

        # 1. Instantiate the Return Volatility Estimator (RVE)
        self.rve = ReturnVolatilityEstimateNetwork(D, dropout, num_layers, layer_dim)
        
        # 2. Instantiate the Transformer Backbone
        self.backbone = MuBackBone(backbone_dim, self.output_type, num_stocks, dropout, num_layers, layer_dim)
        
        # 3. Register the global expected return buffer
        self.register_buffer('r_est', th.tensor(r_est, dtype=th.float32).view(1, -1, 1))

    def forward(self, x):
        B, K, T, D = x.shape

        # 1. Base Slicing
        l_close = x[:, :, :, self.idx['logret_close']]
        l_open  = x[:, :, :, self.idx['logret_open']]
        l_high  = x[:, :, :, self.idx['logret_high']]
        l_low   = x[:, :, :, self.idx['logret_low']]

        # 2. RVE Routing & Error Construction
        if self.input_type in [BackboneInputType.revol_only, BackboneInputType.hybrid]:
            mu, sigma = self.rve(x, l_close)
            mu_exp = mu.unsqueeze(2)
            s_exp = sigma.unsqueeze(2)

            err_c = (l_close - mu_exp) / s_exp
            err_h = l_high / s_exp
            err_o = (l_open - (mu_exp * self.r_est)) / s_exp
            err_l = l_low / s_exp

            err_map = {
                self.idx['logret_close']: err_c,
                self.idx['logret_high']: err_h,
                self.idx['logret_open']: err_o,
                self.idx['logret_low']: err_l
            }

            if self.input_type == BackboneInputType.revol_only:
                ordered_errors = [err_map[i] for i in sorted(err_map.keys())]
                errors = th.stack(ordered_errors, dim=-1)
            else: 
                errors = x.clone() 
                target_indices = list(err_map.keys())
                target_tensors = th.stack(list(err_map.values()), dim=-1)
                errors[..., target_indices] = target_tensors
        else:
            errors = x
            mu = th.zeros((B, K), device=x.device, dtype=x.dtype)
            sigma = th.ones((B, K), device=x.device, dtype=x.dtype)

        # 4. Backbone pass (Returns a tuple)
        out_tuple = self.backbone(errors)
        
        # 5. Targeted Return Routing
        if self.output_type == ModelOutputType.logreturn:
            # Returns: error_prediction, mu, sigma
            return out_tuple[0], mu, sigma
            
        elif self.output_type == ModelOutputType.rank:
            # Returns: rank_prediction ONLY
            return out_tuple[0]
            
        elif self.output_type == ModelOutputType.effr_vol:
            # Returns: effr_prediction, vol_prediction ONLY
            return out_tuple[0], out_tuple[1]

class RevolSimplifiedLogReturn(nn.Module):
    def __init__(self, r_est: np.ndarray, feature_indices: dict, 
                 num_stocks: int, dropout: float, num_layers: int, layer_dim: int):
        super().__init__()
        self.idx = feature_indices
        
        self.backbone = MuBackBone(
            indim=4, 
            output_type=ModelOutputType.logreturn, 
            num_stocks=num_stocks, 
            dropout=dropout, 
            num_layers=num_layers, 
            layer_dim=layer_dim
        )
        
        self.register_buffer('r_est', th.tensor(r_est, dtype=th.float32).view(1, -1, 1))

    def forward(self, x: th.Tensor):

        l_close = x[..., self.idx['logret_close']]
        l_open  = x[..., self.idx['logret_open']]
        l_high  = x[..., self.idx['logret_high']]
        l_low   = x[...,  self.idx['logret_low']]

        mu = l_close.mean(dim=-1)
        sigma = l_close.std(dim=-1, unbiased=False).clamp(min=1e-8)

        mu_exp = mu.unsqueeze(-1)
        s_exp = sigma.unsqueeze(-1)

        # error generation same as Revol
        err_c = (l_close - mu_exp) / s_exp
        err_h = l_high / s_exp
        err_o = (l_open - (mu_exp * self.r_est)) / s_exp
        err_l = l_low / s_exp

        err_map = {
            self.idx['logret_close']: err_c,
            self.idx['logret_high']: err_h,
            self.idx['logret_open']: err_o,
            self.idx['logret_low']: err_l
        }
        
        ordered_errors = [err_map[i] for i in sorted(err_map.keys())]
        errors = th.stack(ordered_errors, dim=-1)

        out_tuple = self.backbone(errors)
        
        return out_tuple[0], mu, sigma

# 1. Grab the first fold for initialization
first_fold = all_folds_data[0]
train_bundle = first_fold['train']
D = train_bundle['X'].shape[3]

x_sample = th.from_numpy(train_bundle['X'][:batchsize])


# sample model for parameter calculatoin
model = ReVolSuperNetwork(
    input_type= BackboneInputType.revol_only, 
    output_type=ModelOutputType.logreturn, 
    r_est=first_fold['r'], 
    feature_indices=feature_indicies,
    num_stocks=NUM_STOCKS,
    dropout=dropout,
    num_layers=num_layers,
    layer_dim=layer_dim,
    D=D
)

# --- Verification Pass ---
# Unpack appropriately depending on the current Enum setup
if model_output_type == ModelOutputType.logreturn:
    error_out, mu_out, sigma_out = model(x_sample)
elif model_output_type == ModelOutputType.rank:
    rank_out = model(x_sample)
elif model_output_type == ModelOutputType.effr_vol:
    effr_out, vol_out = model(x_sample)

summary(model, input_data=(x_sample), verbose=1)
print("Initialization successful. Model is ready for training.")



# 5. Optimization Objectives & Metrics
Establishes the specialized loss functions required for quantitative finance (Continuous MSE with Drift Guidance, Pairwise RankNet BCE) and defines the evaluation metrics (Information Coefficient (IC) and Rank Information Coefficient (RIC)).

In [ ]:
"""
Predefining for Training Loop
"""
def logret_pred_reconstruction(error_pred, musigma_pred, sigma_pred):
    """
    Reconstructs the final predicted log-returns at time T+1.
    Formula: logret_pred = musigma_pred + (error_pred * sigma_pred)
    """
    # 1. Slice to ensure alignment with the stock universe
    mu_stocks = musigma_pred[:, :NUM_STOCKS]
    sigma_stocks = sigma_pred[:, :NUM_STOCKS]
    error_stocks = error_pred[:, :NUM_STOCKS]
    
    # 2. Reconstruct following the GBM identity
    logret_pred = mu_stocks + (error_stocks * sigma_stocks)
    
    return logret_pred

def logret_loss_function(logret_true, error_pred, musigma_pred, sigma_pred, rc):
    # 1. Reconstruction & Slicing
    logret_true_stocks = logret_true[:, :NUM_STOCKS]
    logret_pred = logret_pred_reconstruction(error_pred, musigma_pred, sigma_pred)
    mu_p = musigma_pred[:, :NUM_STOCKS]

    # 2. Continuous Return MSE Loss
    return_loss = F.mse_loss(logret_pred, logret_true_stocks)
    
    # 3. Drift Loss for Guidance
    rc_stocks = rc[:, :NUM_STOCKS, :] 
    w_mean = rc_stocks.mean(dim=2)
    w_std = rc_stocks.std(dim=2).clamp(min=1e-8)
    drift_loss = F.mse_loss(mu_p / w_std, w_mean / w_std)

    return return_loss, drift_loss

def effr_pred_reconstruction(effr_pred, vol_pred):
    """
    Reconstructs the log-returns using the EFFR and Volatility heads.
    """
    # 1. Slice to ensure alignment with the stock universe and multiply
    effr_stocks = effr_pred[:, :NUM_STOCKS]
    vol_stocks = vol_pred[:, :NUM_STOCKS]
    
    return effr_stocks * vol_stocks

def effr_loss_function(logret_true, effr_pred, vol_pred):
    # 1. Reconstruction & Slicing
    logret_true_stocks = logret_true[:, :NUM_STOCKS]
    logret_pred = effr_pred_reconstruction(effr_pred, vol_pred)
    
    # 2. Continuous Return MSE Loss
    return_loss = F.mse_loss(logret_pred, logret_true_stocks)
    
    return return_loss

def rank_loss_function(logret_true, rank_logits_pred):
    """
    Pairwise Binary Cross Entropy (BCE) loss for ranking.
    Penalizes the model if it assigns a higher logit score to a stock 
    that actually had a lower forward return.
    """
    # 1. Slice out only the stocks (ignore index/macro features)
    y_true_stocks = logret_true[:, :NUM_STOCKS] 
    rank_logits_stocks = rank_logits_pred[:, :NUM_STOCKS]
    
    # 2. Compute pairwise differences across the NUM_STOCKS dimension
    diff_true = y_true_stocks.unsqueeze(2) - y_true_stocks.unsqueeze(1)
    diff_pred = rank_logits_stocks.unsqueeze(2) - rank_logits_stocks.unsqueeze(1)
    
    # 3. Create binary labels: 1 if true return i > true return j, else 0
    labels = (diff_true > 0).float()
    
    # 4. Mask out lower triangle and exact ties to prevent double counting
    tri_mask = th.triu(th.ones((NUM_STOCKS, NUM_STOCKS), dtype=th.bool, device=logret_true.device), diagonal=1)
    non_tie_mask = (diff_true.abs() > 1e-7) 
    valid_pairs_mask = tri_mask.unsqueeze(0) & non_tie_mask
    
    # 5. Extract valid pairs and calculate BCE loss
    logits = diff_pred[valid_pairs_mask]
    targets = labels[valid_pairs_mask]
    loss = F.binary_cross_entropy_with_logits(logits, targets)
    
    return loss

def calculate_ic(logret_true, logret_pred):
    """
    Calculates the Pearson Information Coefficient (IC) per timestamp in the batch.
    Returns a 1D tensor of IC values of shape [B].
    """
    # 1. Align to stock universe
    true_v = logret_true[:, :NUM_STOCKS]
    pred_v = logret_pred[:, :NUM_STOCKS]
    
    # 2. Mean-center cross-sectionally
    x_c = pred_v - pred_v.mean(dim=1, keepdim=True)
    y_c = true_v - true_v.mean(dim=1, keepdim=True)
    
    # 3. Compute correlation
    return F.cosine_similarity(x_c, y_c, dim=1)

def calculate_ric(logret_true, rank_logits_pred):
    """
    Calculates the Spearman Rank Information Coefficient (RIC) per timestamp in the batch.
    Returns a 1D tensor of RIC values of shape [B].
    """
    # 1. Align to stock universe
    true_v = logret_true[:, :NUM_STOCKS]
    pred_logits = rank_logits_pred[:, :NUM_STOCKS]
    
    # 2. Convert continuous returns and abstract logits to discrete integer ranks
    true_ranks = true_v.argsort(dim=1).argsort(dim=1).float()
    pred_ranks = pred_logits.argsort(dim=1).argsort(dim=1).float()
    
    # 3. Mean-center cross-sectionally
    x_c = pred_ranks - pred_ranks.mean(dim=1, keepdim=True)
    y_c = true_ranks - true_ranks.mean(dim=1, keepdim=True)
    
    # 4. Compute rank correlation
    return F.cosine_similarity(x_c, y_c, dim=1)


class GeometricSchedule:
    def __init__(self, init_lr: float, final_lr: float):
        self.init_lr = init_lr
        self.final_lr = final_lr
        self.lr = None

    def __call__(self, progress: float) -> float:
        self.lr = self.init_lr * (self.final_lr / self.init_lr) ** progress
        return self.lr

    def __repr__(self) -> str:
        return f"GeometricSchedule(init_lr={self.init_lr}, final_lr={self.final_lr})"

schedule = GeometricSchedule(lr_start, lr_end)

def lr_lambda(epoch):
    return schedule(epoch / epochs) 


def set_run_seed(seed: int):
    """Resets the RNG state for a reproducible run."""
    np.random.seed(seed)
    th.manual_seed(seed)
    th.cuda.manual_seed_all(seed)
    return np.random.default_rng(seed)


# 6. Walk-Forward Training Pipeline
Contains the core training loop (`run_walk_forward_optimization`). It manages warm-starts across WFO folds, applies early stopping based on validation IC/RIC, and handles Isotonic Regression mapping for rank-to-return projection.

In [ ]:
def run_walk_forward_optimization(
    all_folds_data, 
    model, 
    output_type: ModelOutputType, 
    seed: int,
    save_dir: Path
):
    # 1. Lock determinism and get RNG for batching (still needed for batch sampling)
    rng = set_run_seed(seed)
    
    # 2. Directory Setup
    os.makedirs(save_dir, exist_ok=True)
    exp_name = save_dir.name # Pulls the folder name for logging purposes
    
    # Dynamic metric name for logging
    metric_name = "RIC" if output_type == ModelOutputType.rank else "IC"
    
    print(f"\n{'='*60}")
    print(f"Starting Experiment: {exp_name} on {device}")
    print(f"Strategy: Warm Start | Early Stopping: {metric_name}-based | Total Folds: {len(all_folds_data)}")
    print(f"Models & History will be saved to: {save_dir}")
    print(f"{'='*60}")

    wfo_history = {
        "fold_final_val_loss": [],
        "fold_final_metric_mean": [],
        "fold_final_metric_std": [],
        "fold_epochs_run": [],
        "fold_pre_train_metric": []
    }

    # Ensure model is on the correct device
    model.to(device)

    # --- Outer Loop: Iterate through Time ---
    for fold_idx, fold in enumerate(all_folds_data):
        
        # ---------------------------------------------------------
        # 0. DATA PREPARATION
        # ---------------------------------------------------------
        train_tensors = {
            k: th.as_tensor(v, device=device, dtype=th.float32) 
            for k, v in fold['train'].items() if k in ['X', 'logret_close']
        }
        val_tensors = {
            k: th.as_tensor(v, device=device, dtype=th.float32) 
            for k, v in fold['val'].items() if k in ['X', 'logret_close']
        }

        N_train = train_tensors['X'].shape[0]
        N_val   = val_tensors['X'].shape[0]

        # ---------------------------------------------------------
        # 1. PRE-TRAINING VALIDATION (Metric-Focused)
        # ---------------------------------------------------------
        if fold_idx > 0: 
            model.eval()
            pre_val_metrics = [] 
            
            with th.inference_mode():
                for start in range(0, N_val, eval_batchsize):
                    end = min(start + eval_batchsize, N_val)
                    xb_v = val_tensors['X'][start:end]
                    
                    # --- Calculate IC / RIC Routing ---
                    if output_type == ModelOutputType.logreturn:
                        err_v, mu_v, sig_v = model(xb_v)
                        pred_v = logret_pred_reconstruction(err_v, mu_v, sig_v)
                        batch_metrics = calculate_ic(val_tensors['logret_close'][start:end], pred_v)
                        
                    elif output_type == ModelOutputType.effr_vol:
                        effr_v, vol_v = model(xb_v)
                        pred_v = effr_pred_reconstruction(effr_v, vol_v)
                        batch_metrics = calculate_ic(val_tensors['logret_close'][start:end], pred_v)
                        
                    elif output_type == ModelOutputType.rank:
                        rank_logits_v = model(xb_v)
                        batch_metrics = calculate_ric(val_tensors['logret_close'][start:end], rank_logits_v)
                        
                    pre_val_metrics.append(batch_metrics)
            
            # Aggregate and Display
            all_pre_metrics = th.cat(pre_val_metrics).cpu().numpy()
            pre_train_ic_mean = np.mean(all_pre_metrics)
            wfo_history["fold_pre_train_metric"].append(float(pre_train_ic_mean))
            print(f"\n[Pre-Check] Fold {fold_idx+1} | Prev Model Zero-Shot {metric_name}: {pre_train_ic_mean:.4f} (±{np.std(all_pre_metrics):.4f})")
        else:
            wfo_history["fold_pre_train_metric"].append(0.0)

        # ---------------------------------------------------------
        # 2. MODEL & OPTIMIZER SETUP
        # ---------------------------------------------------------
        # As long as your new models have an `r_est` attribute, this will work seamlessly
        model.r_est.copy_(th.as_tensor(fold['r'], dtype=th.float32, device=device).view(1,-1,1))

        # Warm Start: Keep weights, reset optimizer/scheduler
        opt = th.optim.AdamW(model.parameters(), lr=lr_start, weight_decay=l2_norm)
        scheduler = th.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lr_lambda)

        best_val_metric_mean = -float('inf')
        best_val_loss_at_best_ic = float('inf')
        best_metric_stats = (0.0, 0.0) 
        best_model_state = None
        patience_counter = 0
        
        # ---------------------------------------------------------
        # 3. TRAINING LOOP
        # ---------------------------------------------------------
        print(f"\n>> FOLD {fold_idx+1}/{len(all_folds_data)} [Train: {N_train} | Val: {N_val}]")
        epochs_run = 0
        
        for ep in range(epochs):
            model.train()
            batch_losses = []
            
            # --- A. Training Step ---
            for _ in range(iters):
                idx = rng.integers(0, N_train, batchsize)
                xb = train_tensors['X'][idx]
                logret_true_b = train_tensors['logret_close'][idx]
                
                # Dynamic Training Routing
                if output_type == ModelOutputType.logreturn:
                    error_p, mu_p, sig_p = model(xb)
                    rc = xb[..., IDX_LOGRET_CLOSE]
                    l_ret, l_drift = logret_loss_function(logret_true_b, error_p, mu_p, sig_p, rc)
                    loss = return_loss_coeff * l_ret + drift_loss_coeff * l_drift
                    
                elif output_type == ModelOutputType.effr_vol:
                    effr_p, vol_p = model(xb)
                    loss = effr_loss_function(logret_true_b, effr_p, vol_p)
                    
                elif output_type == ModelOutputType.rank:
                    rank_logits_p = model(xb)
                    loss = rank_loss_function(logret_true_b, rank_logits_p)
                
                opt.zero_grad()
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())

            scheduler.step()
            train_loss = np.mean(batch_losses)
            
            # --- B. Validation Step ---
            model.eval()
            val_losses = []
            val_metrics = [] 
            
            with th.inference_mode():
                for start in range(0, N_val, eval_batchsize):
                    end = min(start + eval_batchsize, N_val)
                    xb_val = val_tensors['X'][start:end]
                    lret_true_v = val_tensors['logret_close'][start:end]
                    
                    # Dynamic Validation Routing
                    if output_type == ModelOutputType.logreturn:
                        err_v, mu_v, sig_v = model(xb_val)
                        rc_v = xb_val[..., IDX_LOGRET_CLOSE] 
                        r_l, d_l = logret_loss_function(lret_true_v, err_v, mu_v, sig_v, rc_v)
                        val_losses.append(return_loss_coeff * r_l.item() + drift_loss_coeff * d_l.item())
                        
                        pred_v = logret_pred_reconstruction(err_v, mu_v, sig_v)
                        batch_metrics = calculate_ic(lret_true_v, pred_v)
                        
                    elif output_type == ModelOutputType.effr_vol:
                        effr_v, vol_v = model(xb_val)
                        r_l = effr_loss_function(lret_true_v, effr_v, vol_v)
                        val_losses.append(r_l.item())
                        
                        pred_v = effr_pred_reconstruction(effr_v, vol_v)
                        batch_metrics = calculate_ic(lret_true_v, pred_v)
                        
                    elif output_type == ModelOutputType.rank:
                        rank_logits_v = model(xb_val)
                        r_l = rank_loss_function(lret_true_v, rank_logits_v)
                        val_losses.append(r_l.item())
                        
                        batch_metrics = calculate_ric(lret_true_v, rank_logits_v)
                        
                    val_metrics.append(batch_metrics)
                    
            val_loss_epoch = np.mean(val_losses)
            val_metric_mean = np.mean(th.cat(val_metrics).cpu().numpy())
            val_metric_std = np.std(th.cat(val_metrics).cpu().numpy())
            
            # --- C. Early Stopping Check (IC/RIC Based with Burn-in) ---
            marker = ""
            if fold_idx == 0 and ep < burn_in_epochs: 
                best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                marker = "~"
            else: 
                if val_metric_mean > best_val_metric_mean:
                    best_val_metric_mean = val_metric_mean
                    best_val_loss_at_best_ic = val_loss_epoch
                    best_metric_stats = (val_metric_mean, val_metric_std)
                    best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    patience_counter = 0
                    marker = "*" 
                else:
                    patience_counter += 1

            print(f"Ep {ep:02d} | Train: {train_loss:.5f} | Val: {val_loss_epoch:.5f} | {metric_name}: {val_metric_mean:.4f} (±{val_metric_std:.4f}) {marker}")

            if patience_counter >= patience: 
                print(f"Early stopping triggered at epoch {ep}")
                epochs_run = ep
                break
                
        # ---------------------------------------------------------
        # 4. FINALIZE FOLD (Save & Clean)
        # ---------------------------------------------------------
        if best_model_state:
            model.load_state_dict(best_model_state)
        
        fold_filename = save_dir / f"model_fold_{fold_idx+1}.pth"
        th.save(model.state_dict(), fold_filename)
        
        wfo_history["fold_final_val_loss"].append(float(best_val_loss_at_best_ic))
        wfo_history["fold_final_metric_mean"].append(float(best_val_metric_mean))
        wfo_history["fold_final_metric_std"].append(float(best_metric_stats[1]))
        wfo_history["fold_epochs_run"].append(int(epochs_run))

        del train_tensors, val_tensors
        th.cuda.empty_cache()
        
        print(f"Fold {fold_idx+1} Saved to {fold_filename} | Best Val {metric_name}: {best_val_metric_mean:.4f}")

    # ---------------------------------------------------------
    # 5. EXPORT HISTORY
    # ---------------------------------------------------------
    history_file = save_dir / "wfo_history.json"
    with open(history_file, 'w') as f:
        json.dump(wfo_history, f, indent=4)
        
    print(f"\n--- Walk-Forward Optimization Complete for {exp_name} ---")
    return wfo_history

def fit_and_save_isotonic_mapping(train_logits, train_returns, save_path: Path):
    """
    Fits a global Isotonic Regressor and saves the model to disk using pickle.
    """
    global_train_logits = train_logits.flatten()
    global_train_returns = train_returns.flatten()
    
    ir = IsotonicRegression(out_of_bounds='clip')
    ir.fit(global_train_logits, global_train_returns)
    
    with open(save_path, 'wb') as f:
        pickle.dump(ir, f)
    print(f"  -> Isotonic mapping saved to: {save_path.name}")

def load_and_predict_isotonic(test_logits, load_path: Path):
    """
    Loads a saved global Isotonic Regressor using pickle and maps test logits
    to expected continuous returns while preserving cross-sectional rank.
    
    Args:
        test_logits (np.ndarray): (T_test, N) Rank logits from the inference set.
        load_path (Path): The file path where the model is saved (.pkl).
        
    Returns:
        np.ndarray: (T_test, N) Mapped expected log returns ready for Markowitz.
    """
    # Load the fitted scikit-learn model from disk using read-binary mode
    with open(load_path, 'rb') as f:
        ir = pickle.load(f)
    
    T_test, N = test_logits.shape
    
    # Predict on the flattened logits and reshape back to the (Time, Stocks) matrix
    mapped_expected_returns = ir.predict(test_logits.flatten()).reshape(T_test, N)
        
    return mapped_expected_returns



In [ ]:
# # # --- Define Seeds ---
seeds = [10, 20, 30] 

# 7. Ablation Grid Execution
Iterates through the 9-configuration ablation matrix (Input Routings $\times$ Output Targets) across multiple random seeds, training the SuperNetwork to isolate the impact of the ReVol information bottleneck.

In [ ]:
# 1. Generate all 27 combinations in the exact fail-fast order
all_configs = list(itertools.product(seeds, BackboneInputType, ModelOutputType))
my_configs = all_configs

total_runs = len(my_configs)
print(f"Initiating Notebook 1: Processing {total_runs} configurations...")

# Extract feature dimension once before the loop
D = all_folds_data[0]['train']['X'].shape[3]

# 2. Iterate through the assigned chunk
for current_run, (seed, input_type, output_type) in enumerate(my_configs, 1):
    print(f"\n\n{'#'*80}")
    print(f"### NOTEBOOK 1 PROGRESS: Run {current_run} / {total_runs}")
    print(f"### CONFIG: Seed={seed} | Input={input_type.name} | Output={output_type.name}")
    print(f"{'#'*80}")
    
    # --- Define and construct the output directory ---
    exp_name = f"{input_type.name}_{output_type.name}_seed{seed}"
    save_dir = Path.cwd().parent / 'Models' / exp_name
    
    # --- Instantiate the Model ---
    # CRITICAL: Lock the seed right before instantiating the model to ensure 
    # the randomly initialized weights are deterministic per run.
    set_run_seed(seed) 
    
    current_model = ReVolSuperNetwork(
        input_type=input_type, 
        output_type=output_type, 
        r_est=all_folds_data[0]['r'], 
        feature_indices=feature_indicies, 
        num_stocks=NUM_STOCKS,
        dropout=dropout,
        num_layers=num_layers,
        layer_dim=layer_dim,
        D=D
    )
    
    # Execute the Walk-Forward Optimization
    run_walk_forward_optimization(
        all_folds_data=all_folds_data,
        model=current_model,
        output_type=output_type,
        seed=seed,
        save_dir=save_dir
    )

print("\n\n" + "="*80)
print(f"NOTEBOOK 1 COMPLETED {total_runs} EXPERIMENTS SUCCESSFULLY!")
print("="*80)

In [ ]:
""" Isotonic Mappings save for rank prediction dirs"""

output_type = ModelOutputType.rank
for seed in seeds:
    for input_type in BackboneInputType:
        
        # 1. Locate the specific experiment directory
        exp_name = f"{input_type.name}_{output_type.name}_seed{seed}"
        save_dir = Path.cwd().parent / 'Models' / exp_name
        
        if not save_dir.exists():
            print(f"Skipping {exp_name} (Directory not found)")
            continue
            
        print(f"\nProcessing Experiment: {exp_name}")
        
        # 2. Iterate through all folds
        for fold_idx, fold in enumerate(all_folds_data):
            f_num = fold_idx + 1
            model_path = save_dir / f"model_fold_{f_num}.pth"
            iso_save_path = save_dir / f"model_fold_{f_num}_iso.pkl"
            
            if not model_path.exists():
                print(f"  Missing model for Fold {f_num}, skipping...")
                continue
                
            # 3. Instantiate and load the trained model
            D = fold['train']['X'].shape[3]
            model = ReVolSuperNetwork(
                input_type=input_type, 
                output_type=output_type, 
                r_est=fold['r'], 
                feature_indices=feature_indicies,
                num_stocks=NUM_STOCKS,
                dropout=dropout,
                num_layers=num_layers,
                layer_dim=layer_dim,
                D=D
            )
            model.load_state_dict(th.load(model_path, map_location=device))
            model.to(device)
            model.eval()
            
            # 4. Extract Training Data
            X_train = th.as_tensor(fold['train']['X'], device=device, dtype=th.float32)
            y_train = fold['train']['logret_close'][:, :NUM_STOCKS]  # Keep as numpy array for sklearn
            N_train = X_train.shape[0]
            
            # 5. Safely batch the training data to get rank logits
            train_logits_list = []
            with th.inference_mode():
                for start in range(0, N_train, eval_batchsize):
                    end = min(start + eval_batchsize, N_train)
                    xb = X_train[start:end]
                    
                    # Forward pass for rank model
                    rank_logits = model(xb)
                    train_logits_list.append(rank_logits.cpu().numpy())
            
            # Concatenate all batched logits into (T_train, N)
            full_train_logits = np.concatenate(train_logits_list, axis=0)
            
            # 6. Fit and save the Isotonic mapping
            fit_and_save_isotonic_mapping(full_train_logits, y_train, iso_save_path)

print("\n" + "="*80)
print("Isotonic Mapping Generation Complete!")
print("="*80)



# 8. Downstream Portfolio Optimization (Mean-L2)
Translates the network's directional predictions into executable capital weights using a convex Mean-Variance optimizer. It bounds the $L_1$ norm to allow flexible long/short/cash allocations and simulates the chronological portfolio backtest.

In [ ]:
# # Load models and calculate the money curve

def solve_markowitz(mu_preds, slippage_bps=0.0, spread_bps=0.0, fee_bps=0.0, l2_reg=0.5):
    """
    Simplified Mean-L2 Optimizer.
    Horizon: 30-120 mins.
    """
    T, N = mu_preds.shape
    weights_list = []

    # Combine linear transaction costs
    lin_cost_penalty = spread_bps + fee_bps
    quad_cost_penalty = slippage_bps

    w_prev = np.zeros(N)

    for t in range(T):
        w = cp.Variable(N)
        
        # --- 1. Expected Return ---
        expected_return = mu_preds[t] @ w
            
        # --- 2. Risk (L2 Regularization / Lambda W^T W) ---
        risk_penalty = l2_reg * cp.sum_squares(w)
            
        # --- 3. Transaction Costs ---
        linear_cost = lin_cost_penalty * cp.norm(w - w_prev, 1)
        quadratic_cost = quad_cost_penalty * cp.sum_squares(w - w_prev)
        
        # --- 4. Objective & Constraints ---
        objective = cp.Maximize(expected_return - risk_penalty - linear_cost - quadratic_cost)
        
        # L1 norm <= 1.0 allows long/short flexibility AND cash holding.
        # If sum(|w|) < 1.0, the remainder is cash.
        constraints = [
            cp.norm(w, 1) <= 1.0 
        ]
        
        prob = cp.Problem(objective, constraints)
        
        # MOSEK is highly recommended for quadratic problems, but ECOS/OSQP act as fallbacks
        prob.solve(solver=cp.MOSEK)
        
        curr_w = w.value if w.value is not None else w_prev
        weights_list.append(curr_w)
        w_prev = curr_w
            
    return np.array(weights_list)

# Initialize the master dictionary to store all Backtest DataFrames
final_results_dict = {}

print("="*80)
print("Starting Global WFO Inference & Portfolio Optimization")
print("="*80)

# Iterate over all seeds and all model configurations
for seed in seeds:
    for input_type in BackboneInputType:
        for output_type in ModelOutputType:
            
            # 1. Locate the specific experiment directory
            exp_name = f"{input_type.name}_{output_type.name}_seed{seed}"
            save_dir = Path.cwd().parent / 'Models' / exp_name
            
            if not save_dir.exists():
                print(f"Skipping {exp_name} (Directory not found)")
                continue
                
            print(f"\n[{exp_name}] Processing Inference & Allocation...")
            all_fold_results = []
            
            # 2. Instantiate Model
            D = all_folds_data[0]['train']['X'].shape[3]
            model = ReVolSuperNetwork(
                input_type=input_type, 
                output_type=output_type, 
                r_est=all_folds_data[0]['r'], 
                feature_indices=feature_indicies,
                num_stocks=NUM_STOCKS,
                dropout=dropout,
                num_layers=num_layers,
                layer_dim=layer_dim,
                D=D
            )
            model.to(device)
            
            # 3. Iterate through Folds
            for fold_idx, fold in enumerate(all_folds_data):
                f_num = fold_idx + 1
                
                model_path = save_dir / f"model_fold_{f_num}.pth"
                iso_load_path = save_dir / f"model_fold_{f_num}_iso.pkl"
                
                if not model_path.exists():
                    print(f"  Missing model for Fold {f_num}, skipping...")
                    continue
                
                # Load weights and update regime constraints
                model.load_state_dict(th.load(model_path, map_location=device))
                model.r_est.copy_(th.as_tensor(fold['r'], dtype=th.float32, device=device).view(1,-1,1))
                model.eval()
                
                # Extract Test Tensors
                test_data = fold['test']
                X_test = th.as_tensor(test_data['X'], device=device, dtype=th.float32)
                
                # 4. Inference Routing
                with th.inference_mode():
                    if output_type == ModelOutputType.logreturn:
                        err_v, mu_v, sig_v = model(X_test)
                        mu_preds_fold = logret_pred_reconstruction(err_v, mu_v, sig_v).cpu().numpy()
                        
                    elif output_type == ModelOutputType.effr_vol:
                        effr_v, vol_v = model(X_test)
                        mu_preds_fold = effr_pred_reconstruction(effr_v, vol_v).cpu().numpy()
                        
                    elif output_type == ModelOutputType.rank:
                        rank_logits = model(X_test).cpu().numpy()
                        mu_preds_fold = load_and_predict_isotonic(rank_logits, iso_load_path)

                # 5. Solve Portfolio Optimization
                fold_weights = solve_markowitz(
                    mu_preds_fold, 
                    slippage_bps=slippage_bps, 
                    spread_bps=spread_bps, 
                    fee_bps=fee_bps,
                    l2_reg=markowitz_reg
                )

                # 6. Prepare Fold DataFrame
                fold_df_data = {}
                for i, sym in enumerate(DJ30_TICKERS):
                    fold_df_data[f"{sym}_next_weight"] = fold_weights[:, i]
                    fold_df_data[f"{sym}_curr_close"] = test_data['close_curr'][:, i]
                    fold_df_data[f"{sym}_next_close"] = test_data['close_next'][:, i]
                    
                    # Store the raw predicted return for magnitude analysis
                    fold_df_data[f"{sym}_pred_logret"] = mu_preds_fold[:, i] 
                
                df_fold = pd.DataFrame(fold_df_data)
                df_fold['fold'] = f_num
                all_fold_results.append(df_fold)

            # 7. Concatenate all folds for this specific configuration
            backtest_df = pd.concat(all_fold_results, axis=0).reset_index(drop=True)
            final_results_dict[exp_name] = backtest_df
            print(f"  -> Successfully compiled backtest data for {exp_name} | Shape: {backtest_df.shape}")

print("\n" + "="*80)
print(f"Total Configurations Processed: {len(final_results_dict)}")
print("Dictionary `final_results_dict` is now ready for downstream analysis.")
print("="*80)

In [ ]:
# --- Configuration & Targets ---
target_input = BackboneInputType.revol_only
target_output = ModelOutputType.logreturn

ticker = 'MCD'
ticker_idx = DJ30_TICKERS.index(ticker)

# Accumulators for Global Statistical Proof
all_mu_corrs = []
all_sig_corrs = []

print("="*80)
print(f"Running Deterministic Redundancy Diagnostics on RVE...")
print("="*80)

# --- Global Inference Extraction Loop ---
for seed_idx, seed in enumerate(seeds):
    exp_name = f"{target_input.name}_{target_output.name}_seed{seed}"
    save_dir = Path.cwd().parent / 'Models' / exp_name
    
    D = all_folds_data[0]['train']['X'].shape[3]
    model = ReVolSuperNetwork(
        input_type=target_input, 
        output_type=target_output, 
        r_est=all_folds_data[0]['r'], 
        feature_indices=feature_indicies,
        num_stocks=NUM_STOCKS,
        dropout=dropout,
        num_layers=num_layers,
        layer_dim=layer_dim,
        D=D
    )
    model.to(device)

    # Temporary storage to concatenate this specific seed's folds
    seed_mu_rve = []
    seed_mu_simple = []
    seed_sig_rve = []
    seed_sig_simple = []

    for fold_idx, fold in enumerate(all_folds_data):
        f_num = fold_idx + 1
        model_path = save_dir / f"model_fold_{f_num}.pth"
            
        model.load_state_dict(th.load(model_path, map_location=device))
        model.r_est.copy_(th.as_tensor(fold['r'], dtype=th.float32, device=device).view(1,-1,1))
        model.eval()
        
        test_data = fold['test']
        X_test = th.as_tensor(test_data['X'], device=device, dtype=th.float32)
        
        with th.inference_mode():
            # 1. Extract RVE network predictions
            err_v, mu_v, sig_v = model(X_test)
            
            # 2. Reconstruct Simple (Deterministic) SMA/StdDev
            rc = X_test[..., IDX_LOGRET_CLOSE] # Shape: (B, K, T)
            mu_simple = rc.mean(dim=2) 
            diff = rc - mu_simple.unsqueeze(2)
            var_simple = (diff**2).mean(dim=2)
            sig_simple = th.sqrt(var_simple.clamp(min=1e-8))
            
            # 3. Store batch data for concatenation (Shape: B, K)
            seed_mu_rve.append(mu_v.cpu().numpy())
            seed_mu_simple.append(mu_simple.cpu().numpy())
            seed_sig_rve.append(sig_v.cpu().numpy())
            seed_sig_simple.append(sig_simple.cpu().numpy())
        
    # 4. Concatenate all folds to create the continuous test time series for this seed
    # Shape becomes: (Total_Test_Batches, K_Stocks)
    seed_mu_rve_cat = np.concatenate(seed_mu_rve, axis=0)
    seed_mu_simple_cat = np.concatenate(seed_mu_simple, axis=0)
    seed_sig_rve_cat = np.concatenate(seed_sig_rve, axis=0)
    seed_sig_simple_cat = np.concatenate(seed_sig_simple, axis=0)
    
    # 5. Calculate Pearson correlation per stock, then append to global list
    K_stocks = seed_mu_rve_cat.shape[1]
    for k in range(K_stocks):
        corr_mu, _ = pearsonr(seed_mu_rve_cat[:, k], seed_mu_simple_cat[:, k])
        corr_sig, _ = pearsonr(seed_sig_rve_cat[:, k], seed_sig_simple_cat[:, k])
        all_mu_corrs.append(corr_mu)
        all_sig_corrs.append(corr_sig)
        
    # 6. Extract ticker specific continuous time series from the first valid seed for plotting
    if seed_idx == 0:
        ticker_mu_rve_plot = seed_mu_rve_cat[:, ticker_idx]
        ticker_mu_simple_plot = seed_mu_simple_cat[:, ticker_idx]
        ticker_sig_rve_plot = seed_sig_rve_cat[:, ticker_idx]
        ticker_sig_simple_plot = seed_sig_simple_cat[:, ticker_idx]

# --- Calculate & Print Final Averaged Statistics ---
final_avg_mu_corr = np.mean(all_mu_corrs)
final_avg_sig_corr = np.mean(all_sig_corrs)

min_mu_idx = np.argmin(all_mu_corrs)
min_sig_idx = np.argmin(all_sig_corrs)
num_stocks = len(DJ30_TICKERS)

print(f"\nGlobal Diagnostic Results (Calculated per stock over entire test period, then averaged):")
print(f"Mu    | Average Pearson Correlation: {final_avg_mu_corr:.6f} | Minimum: {all_mu_corrs[min_mu_idx]:.6f} ({DJ30_TICKERS[min_mu_idx % num_stocks]})")
print(f"Sigma | Average Pearson Correlation: {final_avg_sig_corr:.6f} | Minimum: {all_sig_corrs[min_sig_idx]:.6f} ({DJ30_TICKERS[min_sig_idx % num_stocks]})")
print("="*80)

# --- Plot ticker specific continuous diagnostic (All Folds) ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Ticker Mu Plot
ax1.plot(ticker_mu_rve_plot, label=r'Network Output: RVE $\mu$', color='blue', linewidth=1.5, alpha=0.8)
ax1.plot(ticker_mu_simple_plot, label=r'Deterministic Baseline: Simple $\mu$', color='orange', linestyle='--', linewidth=1.5, alpha=0.9)
ax1.set_title(fr"{ticker} - Expected Return ($\mu$) Entire Test Period", fontsize=14, fontweight='bold')
ax1.set_ylabel("Log Return", fontsize=12)
ax1.legend(loc="upper right", fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.5)

# Ticker Sigma Plot
ax2.plot(ticker_sig_rve_plot, label=r'Network Output: RVE $\sigma$', color='red', linewidth=1.5, alpha=0.8)
ax2.plot(ticker_sig_simple_plot, label=r'Deterministic Baseline: Simple $\sigma$', color='green', linestyle='--', linewidth=1.5, alpha=0.9)
ax2.set_title(fr"{ticker} - Volatility ($\sigma$) Entire Test Period", fontsize=14, fontweight='bold')
ax2.set_xlabel("Time (Sequential Test Batches)", fontsize=12)
ax2.set_ylabel("Volatility Scale", fontsize=12)
ax2.legend(loc="upper right", fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
def backtest_portfolio(df, start_capital, slippage_bps=0.0, spread_bps=0.0, fee_bps=0.0):
    """
    Simulates a sequence of horizon-based trades (30-120 min).
    
    Mathematical Logic per step t:
    1. Target Position:  n[t] = (Weight[t] * Equity[t-1]) // Price_Curr[t]
    2. Trade Delta:      dn = n[t] - n[t-1]
    3. Linear Cost:      C_lin = |dn| * Price_Curr[t] * (spread + fee)
    4. Quadratic Cost:   C_slip = (dn^2) * Price_Curr[t] * slippage
    5. Cash Update:      Cash[t] = Cash[t-1] - (dn * Price_Curr[t]) - (C_lin + C_slip)
    6. Horizon Exit:     Equity[t] = Cash[t] + sum(n[t] * Price_Next[t])
    
    The final equity_curve[-1] is the valuation at the end of the testing period.
    """
    tickers = sorted(set(col.split("_")[0] for col in df.columns if col.endswith("_weight")))
    
    cash = start_capital
    shares = {sym: 0 for sym in tickers}
    equity_curve = []
    equity = start_capital

    # bps to decimal
    linear_factor = (spread_bps + fee_bps)
    slippage_factor = slippage_bps

    for t in range(len(df)):
        # 1. Trade Execution at CURRENT price (Entry)
        for sym in tickers:
            curr_p = df.loc[t, f"{sym}_curr_close"]
            
            # Target shares based on model weights and current equity
            target_val = df.loc[t, f"{sym}_next_weight"] * equity
            target_shares = target_val // curr_p
            
            diff_shares = target_shares - shares[sym]
            
            # Costs are incurred at the moment of trade execution (at curr_p)
            costs = (abs(diff_shares) * curr_p * linear_factor) + (diff_shares**2 * curr_p * slippage_factor)
            
            cash -= (diff_shares * curr_p) + costs
            shares[sym] = target_shares

        # 2. Valuation at NEXT price (End of horizon exit)
        current_valuation = sum(shares[sym] * df.loc[t, f"{sym}_next_close"] for sym in tickers)
        equity = cash + current_valuation
        equity_curve.append(float(equity))

    return equity_curve

# --- 1. Setup and Directory ---
results_dir = Path.cwd().parent / 'Results' / 'EquityCurves'
results_dir.mkdir(parents=True, exist_ok=True)

test_start_date = pd.to_datetime(all_folds_data[0]['meta']['test_range'][0])
test_end_date = pd.to_datetime(all_folds_data[-1]['meta']['test_range'][-1])
date_sig = f"{test_start_date.date()} to {test_end_date.date()}"

print("="*80)
print("Starting Backtest Valuations & Cost Simulation...")
print("="*80)

# --- 2. Calculate and Save Equity Curves for All Models ---
for exp_name, df in final_results_dict.items():
    print(f"  -> Simulating trading path for {exp_name}...")
    
    model_equity = backtest_portfolio(
        df,
        start_capital=start_capital,
        slippage_bps=slippage_bps, 
        spread_bps=spread_bps, 
        fee_bps=fee_bps 
    )
    
    # Save current run using exp_name
    save_path = results_dir / f"{exp_name}.csv"
    with open(save_path, 'w+') as file:
        file.write('\n'.join(map(str, model_equity)))

# --- 3. Benchmarks Extraction (SPY/DIA) ---
spy_returns_list = []
dia_returns_list = [] 

spy_idx = ALL_TICKERS.index('SPY')
dia_idx = ALL_TICKERS.index('DIA') 

for fold in all_folds_data:
    test_log_rets = fold['test']['logret_close']
    spy_returns_list.append(test_log_rets[:, spy_idx])
    dia_returns_list.append(test_log_rets[:, dia_idx])

# Concatenate and insert 0 at the start to match equity curve length
spy_proper = np.insert(np.concatenate(spy_returns_list), 0, 0)
dia_proper = np.insert(np.concatenate(dia_returns_list), 0, 0) 

snp500 = np.exp(np.cumsum(spy_proper)) * start_capital
dj30_idx = np.exp(np.cumsum(dia_proper)) * start_capital 

# --- 4. Load All Grouped Seeds & Plotting ---
plt.figure(figsize=(16, 9))

# Find all CSV files in the directory
search_pattern = str(results_dir / "*.csv")
all_files = glob.glob(search_pattern)

# Dictionary to hold lists of runs: { 'InputType_OutputType': [seed10_array, seed20_array, ...] }
grouped_runs = defaultdict(list)

for file_path in all_files:
    # Get filename without extension: e.g., "base_logreturn_seed10"
    base_name = os.path.splitext(os.path.basename(file_path))[0]
    
    # Split out the seed part to group the identical model configurations
    if "_seed" in base_name:
        group_name = base_name.split("_seed")[0] 
        seed_data = np.loadtxt(file_path)
        grouped_runs[group_name].append(seed_data)

# Define a distinct color palette based on the number of unique groups (e.g., 9 combinations)
colors = plt.cm.get_cmap('tab10', len(grouped_runs))

# Iterate through groups to plot
for i, (group_name, runs) in enumerate(grouped_runs.items()):
    color = colors(i)
    group_array = np.array(runs)
    
    # 1. Plot individual seeds as light "spaghetti" lines
    for run in runs:
        plt.plot(run, color=color, alpha=0.15, linewidth=1.0, zorder=1)
    
    # 2. Calculate and plot the Group Mean
    group_mean = np.nanmean(group_array, axis=0)
    plt.plot(group_mean, label=f"Mean: {group_name}", 
             color=color, linewidth=3.0, zorder=5)

# --- 5. Plot Benchmarks ---
plt.plot(snp500, label="S&P 500 (SPY) Buy & Hold", 
         linewidth=2, linestyle="--", color='black', alpha=0.8, zorder=2)
plt.plot(dj30_idx, label="Dow Jones 30 (DIA) Buy & Hold", 
         linewidth=2, linestyle="--", color='royalblue', alpha=0.9, zorder=3)

# --- Final Polish ---
plt.xlabel("Time Step (Test Period)")
plt.ylabel(f"Equity Value ($) [Start: ${start_capital:,.2f}]")
plt.title(f"Walk-Forward Portfolio Performance by Strategy Group\n{date_sig}")
# Move legend outside the main plot area to prevent occluding the equity curves
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=True, fontsize=10) 
plt.grid(True, which='both', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print("="*80)
print("Tear Sheet Generation Complete!")
print("="*80)

# 9. Empirical Analysis & Diagnostics
Aggregates global Walk-Forward metrics, cash-holding frequencies, and prediction magnitudes to empirically validate the presence of "magnitude hallucinations" and evaluate the structural redundancy of the parameterized RVE.

In [ ]:
models_dir = Path.cwd().parent / 'Models'

# Nested dictionary to store raw averages: {output_type: {input_type: {metric: [seed1_val, seed2_val, seed3_val]}}}
raw_stats = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

for output_type in ModelOutputType:
    for input_type in BackboneInputType:
        for seed in seeds:
            exp_name = f"{input_type.name}_{output_type.name}_seed{seed}"
            json_path = models_dir / exp_name / "wfo_history.json"
            
            with open(json_path, 'r') as f:
                data = json.load(f)
                
                # Calculate fold-averages for this specific seed run
                val_mean = np.mean(data['fold_final_metric_mean'])
                val_std = np.mean(data['fold_final_metric_std'])
                
                # For pre_train_metric (Test IC), skip fold 0 since it is initialized to 0.0
                test_mean = np.mean(data['fold_pre_train_metric'][1:]) 
                
                epochs = np.mean(data['fold_epochs_run'])
                
                # Append to our stats dictionary
                raw_stats[output_type.name][input_type.name]['val_mean'].append(val_mean)
                raw_stats[output_type.name][input_type.name]['val_std'].append(val_std)
                raw_stats[output_type.name][input_type.name]['test_mean'].append(test_mean)
                raw_stats[output_type.name][input_type.name]['epochs'].append(epochs)

print("="*80)
print("Diagnostic Tables: Input Architecture Comparison")
print("="*80)

# Generate and print a table for each Output Type
for out_name, inputs_data in raw_stats.items():
    
    table_data = []
    
    for in_name, metrics in inputs_data.items():
        # Average across the 3 seeds
        avg_val_mean = np.mean(metrics['val_mean'])
        avg_val_std = np.mean(metrics['val_std'])
        avg_test_mean = np.mean(metrics['test_mean'])
        avg_epochs = np.mean(metrics['epochs'])
        
        # Calculate Information Ratio (IR)
        ir = avg_val_mean / avg_val_std if avg_val_std > 0 else 0.0
        
        table_data.append({
            "Input Arch": in_name,
            "Val Mean (IC/RIC)": avg_val_mean,
            "Val Std": avg_val_std,
            "Val IR": ir,
            "Test Mean (Zero-Shot)": avg_test_mean,
            "Avg Epochs": avg_epochs
        })
        
    # Convert to DataFrame for clean formatting
    df_out = pd.DataFrame(table_data)
    
    # Sort by Test Mean to immediately highlight the best generalizing model
    df_out = df_out.sort_values(by="Test Mean (Zero-Shot)", ascending=False).reset_index(drop=True)
    
    metric_label = "RIC" if out_name == "rank" else "IC"
    print(f"\nTarget Output: {out_name.upper()} (Metric: {metric_label})")
    print("-" * 80)
    # Print formatted to 4 decimal places for readability
    print(df_out.to_string(formatters={
        "Val Mean (IC/RIC)": "{:.4f}".format,
        "Val Std": "{:.4f}".format,
        "Val IR": "{:.4f}".format,
        "Test Mean (Zero-Shot)": "{:.4f}".format,
        "Avg Epochs": "{:.1f}".format
    }))
    print("\n")

In [ ]:
table_data = []

# Iterate strictly using your Enums to avoid any string splitting issues
for output_type in ModelOutputType:
    for input_type in BackboneInputType:
        cash_list = []
        
        for seed in seeds:
            # Construct the exact dictionary key
            exp_name = f"{input_type.name}_{output_type.name}_seed{seed}"
            
            df = final_results_dict[exp_name]
            
            # Extract only the weight columns
            weight_cols = [col for col in df.columns if col.endswith('_next_weight')]
            
            # Calculate L1 norm (sum of absolute weights) per timestep
            l1_norm = df[weight_cols].abs().sum(axis=1)
            
            # Cash is 1 - L1_norm. Clip at 0 to avoid minor floating point anomalies.
            cash_fraction = (1.0 - l1_norm).clip(lower=0)
            
            # Average cash held across the entire test period for this specific run
            avg_run_cash = cash_fraction.mean()
            cash_list.append(avg_run_cash)
        
        #  processed runs for this config, average them
        mean_cash_across_seeds = np.mean(cash_list)
        mean_cash_pct = mean_cash_across_seeds * 100
        
        table_data.append({
            'Output Target': output_type.name.upper(),
            'Input Architecture': input_type.name,
            'Cash Held (%)': mean_cash_pct
        })

# Display the results as a clean text table
df_table = pd.DataFrame(table_data)

# Pivot the data so Rows = Output Target, Columns = Input Architecture
pivot_df = df_table.pivot(index='Output Target', columns='Input Architecture', values='Cash Held (%)')

print("="*80)
print("Average % Portfolio Held in Cash")
print("="*80)
# Print the pivoted DataFrame with 2 decimal places and a '%' sign
print(pivot_df.to_string(float_format="{:.2f}%".format))
print("\n" + "="*80)

# 9b. RVE Structural Redundancy Diagnostics
This section empirically tests whether the parameterized Return Volatility Estimator (RVE) is computationally redundant. It extracts the dynamically learned $\mu$ and $\sigma$ tensors from the trained network across the continuous Walk-Forward test period and compares them against a deterministic, unweighted baseline (Simple Moving Average and standard deviation). By calculating the global Pearson correlation and plotting a worst-case constituent (e.g., MCD), this diagnostic proves that the guidance loss forces the complex LSTM/Attention parameters to collapse into a deterministic function, justifying their complete removal in the final scaled architecture.

In [ ]:
table_data = []

# --- 1. Compute True Baseline Reference ---
# Since actual market returns don't change between models/seeds, 
# we can just compute the true distribution once from any valid run.
sample_key = list(final_results_dict.keys())[0]
sample_df = final_results_dict[sample_key]

all_actuals = []
for sym in DJ30_TICKERS:
    curr_close = sample_df[f"{sym}_curr_close"]
    next_close = sample_df[f"{sym}_next_close"]
    # Calculate actual log return: ln(Next / Curr)
    actual_logret = np.log(next_close / curr_close)
    
    # Drop any NaNs that might occur at dataset boundaries
    all_actuals.extend(actual_logret.dropna().values)

all_actuals = np.array(all_actuals)
true_mean_abs = np.mean(np.abs(all_actuals)) * 1e4
true_std = np.std(all_actuals) * 1e4

# --- 2. Compute Model Distributions ---
# Iterate strictly using Enums
for output_type in ModelOutputType:
    for input_type in BackboneInputType:
        seed_mean_abs_preds = []
        seed_std_preds = []
        
        for seed in seeds:
            exp_name = f"{input_type.name}_{output_type.name}_seed{seed}"
            
            if exp_name in final_results_dict:
                df = final_results_dict[exp_name]
                
                # Array to hold all raw predictions for this specific seed run
                all_preds = []
                
                for sym in DJ30_TICKERS:
                    pred_logret = df[f"{sym}_pred_logret"]
                    all_preds.extend(pred_logret.values)
                
                all_preds = np.array(all_preds)
                
                # Take absolute value so the mean doesn't decompose to 0
                abs_preds = np.abs(all_preds)
                
                # Average magnitude over all stocks and all timesteps for this run
                seed_mean_abs_preds.append(np.mean(abs_preds))
                
                # Standard Deviation of raw predictions to show output equality/spread
                seed_std_preds.append(np.std(all_preds))
                
        if seed_mean_abs_preds:
            table_data.append({
                'Output Target': output_type.name.upper(),
                'Input Architecture': input_type.name,
                'Mean Abs Pred (1e-4)': np.mean(seed_mean_abs_preds) * 1e4,
                'Pred StdDev (1e-4)': np.mean(seed_std_preds) * 1e4
            })

df_table = pd.DataFrame(table_data)

# Create two pivot tables for clean display
pivot_mean = df_table.pivot(index='Output Target', columns='Input Architecture', values='Mean Abs Pred (1e-4)')
pivot_std = df_table.pivot(index='Output Target', columns='Input Architecture', values='Pred StdDev (1e-4)')

print("="*80)
print(f"REFERENCE: TRUE MARKET DISTRIBUTION (scaled by 1e-4)")
print(f"True Mean Absolute Return: {true_mean_abs:.3f}")
print(f"True Return Volatility (StdDev): {true_std:.3f}")
print("="*80)

print("\n" + "="*80)
print("Mean Absolute Prediction Magnitude (scaled by 1e-4)")
print("="*80)
print(pivot_mean.to_string(float_format="{:.3f}".format))

print("\n" + "="*80)
print("Prediction Volatility / StdDev (scaled by 1e-4)")
print("="*80)
print(pivot_std.to_string(float_format="{:.3f}".format))
print("\n" + "="*80)

# 10. Final Model: Scaled Deterministic Backbone
Trains and evaluates the finalized architecture based on the ablation findings: replacing the parameterized RVE with a deterministic moving average bottleneck, and scaling the spatial-temporal MuBackBone to a shallow-wide configuration ($L=1$, $d_{model}=256$).

In [ ]:
final_seed = seeds[0]

large_num_layers = 1
large_layer_dim = 256

exp_name = f"DeterministicReVol_Large_logreturn_seed{final_seed}"
save_dir = Path.cwd().parent / 'FinalModels' / exp_name

set_run_seed(final_seed) 

final_model = RevolSimplifiedLogReturn(
    r_est=all_folds_data[0]['r'], 
    feature_indices=feature_indicies, 
    num_stocks=NUM_STOCKS,
    dropout=dropout,
    num_layers=large_num_layers,
    layer_dim=large_layer_dim
)

run_walk_forward_optimization(
    all_folds_data=all_folds_data,
    model=final_model,
    output_type=ModelOutputType.logreturn,
    seed=final_seed,
    save_dir=save_dir
)

In [29]:
""" Printing the train time statistics of the dinal model"""
final_json_path = save_dir / "wfo_history.json"
with open(final_json_path, 'r') as f:
    data = json.load(f)
    
    final_val_mean = np.mean(data['fold_final_metric_mean'])
    final_val_std = np.mean(data['fold_final_metric_std'])
    final_test_mean = np.mean(data['fold_pre_train_metric'][1:]) 
    final_epochs = np.mean(data['fold_epochs_run'])
    final_ir = final_val_mean / final_val_std if final_val_std > 0 else 0.0
    
    final_table_data = [{
        "Model Config": f"Large Det. ReVol (L={large_num_layers}, d={large_layer_dim})",
        "Val Mean (IC)": final_val_mean,
        "Val Std": final_val_std,
        "Val IR": final_ir,
        "Test Mean (Zero-Shot)": final_test_mean,
        "Avg Epochs": final_epochs
    }]
    
    df_final = pd.DataFrame(final_table_data)
    print(df_final.to_string(formatters={
        "Val Mean (IC)": "{:.4f}".format,
        "Val Std": "{:.4f}".format,
        "Val IR": "{:.4f}".format,
        "Test Mean (Zero-Shot)": "{:.4f}".format,
        "Avg Epochs": "{:.1f}".format
    }))
    print("\n")

                    Model Config Val Mean (IC) Val Std Val IR Test Mean (Zero-Shot) Avg Epochs
0  Large Det. ReVol (L=1, d=256)        0.0025  0.2497 0.0101               -0.0003       20.9




# 11. Final Portfolio Backtest & Benchmark Comparison
Executes the downstream Markowitz solver using the predictions from the finalized Large Deterministic model. Tests varying $L_2$ risk penalties against the passive DJ30 and S&P 500 benchmarks to generate the final thesis equity curve.

In [ ]:
print("="*80)
print("Starting Final Inference & Backtest for Scaled Deterministic ReVol (Large)")
print("="*80)

# --- 1. Setup & Model Initialization ---
exp_name = f"DeterministicReVol_Large_logreturn_seed{final_seed}"
save_dir = Path.cwd().parent / 'FinalModels' / exp_name

# Re-instantiate the Large model
model = RevolSimplifiedLogReturn(
    r_est=all_folds_data[0]['r'], 
    feature_indices=feature_indicies, 
    num_stocks=NUM_STOCKS,
    dropout=dropout,
    num_layers=large_num_layers,
    layer_dim=large_layer_dim
)
model.to(device)

# L2 Regularization values to test (Baseline 0.01, Weakened 0.001)
l2_regs = [markowitz_reg, 0.001]
equity_curves = {}

# Cache predictions to avoid redundant model loading/inference
print("Running Model Inference across all folds...")
cached_predictions = {}

for fold_idx, fold in enumerate(all_folds_data):
    f_num = fold_idx + 1
    model_path = save_dir / f"model_fold_{f_num}.pth"
    model.load_state_dict(th.load(model_path, map_location=device))
    model.r_est.copy_(th.as_tensor(fold['r'], dtype=th.float32, device=device).view(1,-1,1))
    model.eval()
    
    X_test = th.as_tensor(fold['test']['X'], device=device, dtype=th.float32)
    
    with th.inference_mode():
        err_v, mu_v, sig_v = model(X_test)
        mu_preds_fold = logret_pred_reconstruction(err_v, mu_v, sig_v).cpu().numpy()
        
    cached_predictions[fold_idx] = mu_preds_fold

# --- 2. Optimization & Backtest Loop (per L2 Regularizer) ---
for reg_val in l2_regs:
    print(f"Solving Markowitz & Backtesting for L2 Penalty: {reg_val}...")
    all_fold_results = []
    
    for fold_idx, fold in enumerate(all_folds_data):
        test_data = fold['test']
        mu_preds_fold = cached_predictions[fold_idx]
        
        # Solve Portfolio Optimization with varying L2
        fold_weights = solve_markowitz(
            mu_preds_fold, 
            slippage_bps=slippage_bps, 
            spread_bps=spread_bps, 
            fee_bps=fee_bps,
            l2_reg=reg_val
        )
        
        # Prepare Fold DataFrame
        fold_df_data = {}
        for i, sym in enumerate(DJ30_TICKERS):
            fold_df_data[f"{sym}_next_weight"] = fold_weights[:, i]
            fold_df_data[f"{sym}_curr_close"]  = test_data['close_curr'][:, i]
            fold_df_data[f"{sym}_next_close"]  = test_data['close_next'][:, i]
            fold_df_data[f"{sym}_pred_logret"] = mu_preds_fold[:, i] 
            
        df_fold = pd.DataFrame(fold_df_data)
        df_fold['fold'] = fold_idx + 1
        all_fold_results.append(df_fold)

    # Concatenate and Backtest
    final_df = pd.concat(all_fold_results, axis=0).reset_index(drop=True)
    
    equity_curves[reg_val] = backtest_portfolio(
        final_df,
        start_capital=start_capital,
        slippage_bps=slippage_bps, 
        spread_bps=spread_bps, 
        fee_bps=fee_bps 
    )

print("Successfully compiled all backtest variations.")

# --- 3. Benchmarks Extraction (SPY/DIA) ---
spy_returns_list = []
dia_returns_list = [] 

spy_idx = ALL_TICKERS.index('SPY')
dia_idx = ALL_TICKERS.index('DIA') 

for fold in all_folds_data:
    test_log_rets = fold['test']['logret_close']
    spy_returns_list.append(test_log_rets[:, spy_idx])
    dia_returns_list.append(test_log_rets[:, dia_idx])

spy_proper = np.insert(np.concatenate(spy_returns_list), 0, 0)
dia_proper = np.insert(np.concatenate(dia_returns_list), 0, 0) 

snp500 = np.exp(np.cumsum(spy_proper)) * start_capital
dj30_idx = np.exp(np.cumsum(dia_proper)) * start_capital 

# --- 4. Final Equity Curve Plotting ---
plt.figure(figsize=(16, 9))

# Plot Benchmark Curves
plt.plot(snp500, label="S&P 500 (SPY) Buy & Hold", 
         linewidth=2, linestyle="--", color='black', alpha=0.8, zorder=2)
plt.plot(dj30_idx, label="Dow Jones 30 (DIA) Buy & Hold", 
         linewidth=2, linestyle="--", color='royalblue', alpha=0.9, zorder=3)

# Plot ReVol Models with varying L2 Penalties
colors = {0.01: 'forestgreen', 0.001: 'darkorange'}
styles = {0.01: '-', 0.001: '-'}

for reg_val, equity in equity_curves.items():
    plt.plot(equity, 
             label=f"Large Deterministic ReVol (L2={reg_val})", 
             color=colors.get(reg_val, 'purple'), 
             linewidth=3.5 if reg_val == 0.001 else 2.5, 
             linestyle=styles.get(reg_val, '-'),
             alpha=0.9,
             zorder=5)

test_start_date = pd.to_datetime(all_folds_data[0]['meta']['test_range'][0])
test_end_date = pd.to_datetime(all_folds_data[-1]['meta']['test_range'][-1])
date_sig = f"{test_start_date.date()} to {test_end_date.date()}"

# Final Polish
plt.xlabel("Time Step (Test Period)")
plt.ylabel(f"Equity Value ($) [Start: ${start_capital:,.2f}]")
plt.title(f"Final Walk-Forward Portfolio Performance (Varying L2 Penalty)\n{date_sig}")
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=True, fontsize=12) 
plt.grid(True, which='both', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()